# 02 — Model comparison on five hearings

**Purpose.** Run the same five hearings through every available model and record
what each costs. Model choice is the largest single decision in this stage and
the one that should be settled first; everything downstream — how many hearings
fit in a job's wall clock, how many jobs the corpus needs — follows from the
throughput measured here.

**Five hearings, spanning the length distribution.** Picked at the 10th, 30th,
50th, 70th and 90th percentiles of word count within the SST sample, so the
comparison covers a short hearing and a long one rather than five of the same
size. The same five go to every model.

**What is measured**

| | |
|---|---|
| Wall clock per hearing, and per 1,000 input words | the number that sizes a job |
| Chunks, model calls, collapse rounds | how much work the document actually caused |
| Final summary length in tokens | must sit under the 512-token cap |
| Summaries that had to be trimmed | a summary cut at the cap and trimmed back is shorter than intended |
| Named speakers surviving into the final summary | attribution is most of what the summary is for |

Quality is **not** measured here. Which model writes the better summary needs the
C-SPAN reference and a held-out set; this notebook answers the cheaper question
first, which is what each model costs to run at all.

**Every model call is logged.** `lib/summarize.CallLog` writes one JSONL line per
call — stage, level, position, a SHA-256 of the exact input, token counts,
elapsed time, and the complete output. The intermediate summaries are the record
of how the final one was arrived at, and they survive the run rather than
existing only inside it.

**Inputs.** `data/raw/02_hearings/hearings_sst.parquet` — from `05`.

**Outputs.** `interim/summaries/compare_{model}/` — the call logs, and a
`comparison.parquet` of the per-hearing measurements.

## Setup

In [1]:
import gc
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, "../lib")

from paths import HEARINGS_REPO, HF_CACHE, summary_run_dir

HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
for line in (Path(".env").read_text().splitlines() if Path(".env").exists() else []):
    if line.startswith("HF_TOKEN="):
        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import summarize as S
from chunking import chunk_turns

# All four fit in bf16 on a 48 GB card with no quantization. Llama-3.1-8B is the
# model the inherited pipeline used, and it ran there in 4-bit.
MODELS = [
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen3.5-4B",
    "google/gemma-4-E4B-it",
    "meta-llama/Llama-3.1-8B-Instruct",
]
N_HEARINGS = 5
CHUNK_TOKENS = int(os.environ.get("CHUNK_TOKENS", 6144))

print(f"models     : {len(MODELS)}")
for name in MODELS:
    print(f"             {name}")
print(f"hearings   : {N_HEARINGS}")
print(f"chunk      : {CHUNK_TOKENS:,} tokens")
print(f"gpu        : {torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"sampling   : {S.GENERATION}   (prompt {S.PROMPT_VERSION})")

models     : 4
             meta-llama/Llama-3.2-3B-Instruct
             Qwen/Qwen3.5-4B
             google/gemma-4-E4B-it
             meta-llama/Llama-3.1-8B-Instruct
hearings   : 5
chunk      : 6,144 tokens
gpu        : NVIDIA A40, 48 GB
sampling   : {'do_sample': True, 'temperature': 0.1}   (prompt hearing-mapreduce-v1)


## Step 1 — The five hearings

Chosen by percentile of word count so the set spans the distribution. Length is
what drives cost: it decides the chunk count, and the chunk count decides how
many model calls the document needs.

In [2]:
sst = pd.read_parquet(HEARINGS_REPO / "hearings_sst.parquet")
QUANTILES = [0.10, 0.30, 0.50, 0.70, 0.90]

picks = []
for q in QUANTILES:
    target = sst["n_words"].quantile(q)
    row = sst.iloc[(sst["n_words"] - target).abs().argsort().iloc[0]]
    picks.append(row)

frame = pd.DataFrame([{
    "q": f"p{int(q * 100)}", "packageId": p["packageId"], "year": p["year"],
    "chamber": p["chamber"], "words": p["n_words"], "turns": p["n_turns"],
    "title": p["title"][:52],
} for q, p in zip(QUANTILES, picks)])
print(frame.to_string(index=False))
print(f"\ntotal input: {frame['words'].sum():,} words")

  q         packageId  year chamber  words  turns                                                title
p10 CHRG-118hhrg56042  2024   HOUSE  16017    111 ENVIRONMENTALISM OFF THE RAILS: HOW CARB WILL CRIPPL
p30 CHRG-111hhrg53691  2009   HOUSE  25694     95 INDEPENDENT AUDIT OF THE NATIONAL AERONAUTICS AND SP
p50 CHRG-114shrg98129  2015  SENATE  36503    166 PRESERVING THE MULTISTAKEHOLDER MODEL OF INTERNET GO
p70 CHRG-113shrg97626  2014  SENATE  47526    325 AGGRESSIVE E-CIGARETTE MARKETING AND POTENTIAL CONSE
p90 CHRG-119shrg62667  2025  SENATE  66486    358 NOMINATION TO THE NATIONAL AERONAUTICS AND SPACE ADM

total input: 192,226 words


## Step 2 — Run every model over all five

Each model is loaded, run, then freed before the next is loaded, so peak memory
is one model rather than four. A model that fails to load is recorded and the
run continues — a licence not yet accepted should not cost the other three.

In [3]:
records = []

for model_name in MODELS:
    short = model_name.split("/")[-1]
    run_dir = summary_run_dir(f"compare_{short}")
    print("=" * 78)
    print(model_name)
    print("=" * 78)

    try:
        t0 = time.time()
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype="auto", device_map="cuda")
        model.eval()
        load_seconds = time.time() - t0
        params = sum(p.numel() for p in model.parameters()) / 1e9
        print(f"  loaded in {load_seconds:.0f}s — {params:.2f}B params, "
              f"{torch.cuda.memory_allocated() / 1e9:.1f} GB on device")
    except Exception as exc:                                  # noqa: BLE001
        print(f"  UNAVAILABLE: {type(exc).__name__}: {str(exc)[:120]}")
        records.append({"model": short, "packageId": None,
                        "error": f"{type(exc).__name__}: {str(exc)[:200]}"})
        continue

    for q, p in zip(QUANTILES, picks):
        chunks = chunk_turns(p["turns"], tokenizer, CHUNK_TOKENS, spoken_only=True)
        log = S.CallLog(run_dir / "status" / f"{p['packageId']}.jsonl",
                        p["packageId"], model_name)
        t0 = time.time()
        out = S.map_reduce([c["text"] for c in chunks], model, tokenizer,
                           budget_tokens=CHUNK_TOKENS, log=log)
        seconds = time.time() - t0

        named = {s for c in chunks for s in c["speakers"]
                 if s != "Unidentified speaker"}
        kept = {s for s in named if s in out["final_summary"]}
        trimmed = sum(1 for r in log.records
                      if r["stage"] == "reduce"
                      and r["output_tokens"] >= S.REDUCE_MAX_TOKENS - 1)

        records.append({
            "model": short, "q": f"p{int(q * 100)}", "packageId": p["packageId"],
            "words": int(p["n_words"]), "chunks": out["n_chunks"],
            "calls": out["n_calls"], "collapse_rounds": out["n_collapse_rounds"],
            "final_tokens": out["final_tokens"],
            "final_words": len(out["final_summary"].split()),
            "seconds": round(seconds, 1),
            "sec_per_kword": round(seconds / (p["n_words"] / 1000), 2),
            "speakers_named": len(named), "speakers_kept": len(kept),
            "reduce_at_cap": trimmed,
            "summary": out["final_summary"], "error": None,
        })
        print(f"  {f'p{int(q * 100)}':>4} {p['packageId']:<20} "
              f"{p['n_words']:>7,}w  {out['n_chunks']:>3}ch  "
              f"{out['n_calls']:>3}calls  L{out['n_collapse_rounds']}  "
              f"{out['final_tokens']:>3}tok  {seconds:>6.1f}s")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  freed; {torch.cuda.memory_allocated() / 1e9:.1f} GB still allocated\n")

results = pd.DataFrame(records)

meta-llama/Llama-3.2-3B-Instruct


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (47 outputs left out)

   p50 CHRG-114shrg98129     36,503w   10ch   11calls  L0  370tok   130.8s


   p70 CHRG-113shrg97626     47,526w   16ch   17calls  L0  375tok   168.7s


   p90 CHRG-119shrg62667     66,486w   17ch   18calls  L0  346tok   207.5s


  freed; 0.0 GB still allocated



## Step 3 — The comparison

`sec_per_kword` is the number that transfers: multiplied by the sample's total
word count it gives the wall clock for a full run, independent of which hearings
happened to be drawn here.

In [4]:
ok = results.loc[results["error"].isna()].copy()

summary = ok.groupby("model").agg(
    hearings=("packageId", "size"),
    total_seconds=("seconds", "sum"),
    sec_per_kword=("sec_per_kword", "median"),
    calls=("calls", "sum"),
    final_tokens=("final_tokens", "median"),
    at_cap=("reduce_at_cap", "sum"),
    speakers_kept=("speakers_kept", "sum"),
    speakers_named=("speakers_named", "sum"),
).sort_values("sec_per_kword")
summary["speaker_recall"] = (summary["speakers_kept"] /
                             summary["speakers_named"]).map("{:.0%}".format)
print(summary.drop(columns=["speakers_kept", "speakers_named"]).to_string())

# What the measured rate implies for the whole SST sample.
total_kwords = sst["n_words"].sum() / 1000
print(f"\nSST sample: {len(sst):,} hearings, {total_kwords:,.0f}k words")
print("projected wall clock for the full sample, one GPU, sequential:")
for model_name, row in summary.iterrows():
    hours = row["sec_per_kword"] * total_kwords / 3600
    print(f"  {model_name:<26} {hours:6.1f} h    "
          f"({hours / 10:5.1f} h across a 10-way array)")

failed = results.loc[results["error"].notna()]
if len(failed):
    print("\nunavailable:")
    for r in failed.itertuples():
        print(f"  {r.model}: {r.error[:110]}")

                       hearings  total_seconds  sec_per_kword  calls  final_tokens  at_cap speaker_recall
model                                                                                                    
Llama-3.2-3B-Instruct         5          362.0           1.93     59         380.0       0            24%
Llama-3.1-8B-Instruct         5          663.6           3.58     59         375.0       0            31%
Qwen3.5-4B                    5          849.8           4.66     59         431.0       0            45%
gemma-4-E4B-it                5         1110.3           6.08     59         349.0       0            40%

SST sample: 2,352 hearings, 93,644k words
projected wall clock for the full sample, one GPU, sequential:
  Llama-3.2-3B-Instruct        50.2 h    (  5.0 h across a 10-way array)
  Llama-3.1-8B-Instruct        93.1 h    (  9.3 h across a 10-way array)
  Qwen3.5-4B                  121.2 h    ( 12.1 h across a 10-way array)
  gemma-4-E4B-it              158.2 h  

## Step 4 — Read one hearing across the models

The same document summarized by each. Cost is what this notebook measures, but a
model that is fast and writes something useless is not cheap, so the output is
put on the page rather than left to the table.

In [5]:
if len(ok):
    pid = ok["packageId"].iloc[len(ok) // 2]
    row = sst.loc[sst["packageId"] == pid].iloc[0]
    print(f"{pid}  ({row['year']}, {row['chamber']})")
    print(row["title"][:110])
    print(f"{row['n_words']:,} words\n")
    for r in ok.loc[ok["packageId"] == pid].itertuples():
        print("=" * 78)
        print(f"{r.model}   {r.seconds}s   {r.final_tokens} tokens   "
              f"{r.speakers_kept}/{r.speakers_named} speakers")
        print("=" * 78)
        print(r.summary)
        print()

CHRG-118hhrg56042  (2024, HOUSE)
ENVIRONMENTALISM OFF THE RAILS: HOW CARB WILL CRIPPLE THE NATIONAL RAIL NETWORK
16,017 words

Llama-3.2-3B-Instruct   36.2s   358 tokens   2/12 speakers
The congressional hearing on the Environmental Protection Agency's (EPA) proposed regulation requiring all locomotives manufactured after 2035 to operate in a zero-emission configuration when traveling through California was a contentious discussion on the feasibility and impact of the regulation. Chairman Jay Obernolte (R-CA) expressed concerns that the regulation is unrealistic and would cripple the national rail network, citing the lack of existing technology to achieve zero-emission locomotives and the significant costs and disruptions it would cause. In contrast, Ranking Member Zoe Lofgren (D-CA) defended the regulation, stating that it would reduce cancer risk, hospital admissions, and premature deaths by 90%, and that the costs would be offset by the benefits, including $18 billion in health bene

## Step 5 — Persist the measurements

In [6]:
target = summary_run_dir("compare") / "comparison.parquet"
results.to_parquet(target, compression="zstd", index=False)
print(f"wrote {target}  ({len(results)} rows)")

logs = sorted(Path(summary_run_dir("compare").parent).glob("compare_*/status/*.jsonl"))
n_lines = sum(sum(1 for _ in open(p)) for p in logs)
print(f"call logs: {len(logs)} files, {n_lines:,} logged model calls")

wrote /scratch2/acroh/projects/llm_method/data/interim/summaries/compare/comparison.parquet  (20 rows)
call logs: 20 files, 236 logged model calls


## Verification

In [7]:
back = pd.read_parquet(target)
good = back.loc[back["error"].isna()]

assert len(good), "no model produced any result"
print(f"1) {good['model'].nunique()} of {len(MODELS)} models ran, "
      f"{len(good)} hearing-model pairs")

over = good.loc[good["final_tokens"] > S.REDUCE_MAX_TOKENS]
assert len(over) == 0, f"{len(over)} summaries exceed the {S.REDUCE_MAX_TOKENS}-token cap"
print(f"2) every summary is within the {S.REDUCE_MAX_TOKENS}-token cap "
      f"(max {good['final_tokens'].max()})")

empty = good.loc[good["summary"].str.strip().eq("")]
assert len(empty) == 0, f"{len(empty)} empty summaries"
print("3) no empty summaries")

# A summary ending mid-clause would mean the trim guard did not fire.
bad_end = good.loc[~good["summary"].str.rstrip().str.endswith((".", "!", "?", '"', ")"))]
print(f"4) summaries not ending on a sentence boundary: {len(bad_end)}")
for r in bad_end.itertuples():
    print(f"     {r.model} / {r.packageId}: ...{r.summary[-70:]!r}")

# Every call the run made should be on disk, or the log is not a record.
for model_name in good["model"].unique():
    sub = good.loc[good["model"] == model_name]
    d = summary_run_dir(f"compare_{model_name}") / "status"
    logged = sum(sum(1 for _ in open(p)) for p in d.glob("*.jsonl"))
    assert logged == sub["calls"].sum(), (
        f"{model_name}: {logged} lines logged against {sub['calls'].sum()} calls made")
print(f"5) every model call is on disk, all {int(good['calls'].sum()):,} of them")

print("\n6) work per hearing, by model:")
print(good.pivot_table(index="q", columns="model",
                       values="seconds", aggfunc="first").to_string())

1) 4 of 4 models ran, 20 hearing-model pairs
2) every summary is within the 512-token cap (max 467)
3) no empty summaries
4) summaries not ending on a sentence boundary: 0
5) every model call is on disk, all 236 of them

6) work per hearing, by model:
model  Llama-3.1-8B-Instruct  Llama-3.2-3B-Instruct  Qwen3.5-4B  gemma-4-E4B-it
q                                                                              
p10                     62.7                   36.2        78.0            99.4
p30                     93.9                   49.2       119.8           156.1
p50                    130.8                   72.6       164.7           210.5
p70                    168.7                   91.7       230.0           306.8
p90                    207.5                  112.3       257.3           337.5
